In [ ]:
# ============================================================
# Notebook 05 — PB-MMAE Model Development
# Cell 1: Project configuration + frozen 14-channel inventory
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# Project paths
# ------------------------------------------------------------

PROJECT_ROOT = Path(
    r"C:\project 3\Paper1_SSL"
)

PATCH_DIR = (
    PROJECT_ROOT /
    "03_outputs" /
    "patches"
)

PBMMAE_DIR = (
    PROJECT_ROOT /
    "03_outputs" /
    "models" /
    "PBMMAE"
)

PBMMAE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

TABLE_DIR = (
    PROJECT_ROOT /
    "03_outputs" /
    "tables"
)

TABLE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# ------------------------------------------------------------
# Primary experimental configuration
# ------------------------------------------------------------

PATCH_SIZES = [9, 13]

N_FOLDS = 5
N_CHANNELS = 14

# ------------------------------------------------------------
# Frozen 14-channel order
#
# IMPORTANT:
# This order must remain identical to the tensors generated
# in Notebook 03.
# ------------------------------------------------------------

CHANNEL_NAMES = [

    "RTE_TMI",   # 0

    "CBG",       # 1
    "CBG_RES",   # 2
    "HGM",       # 3

    "K",         # 4
    "eTh",       # 5
    "eU",        # 6

    "CPD",       # 7

    "MAG_LD",    # 8
    "GRAV_LD",   # 9
    "DEM_LD",    # 10
    "ID",        # 11

    "DEM",       # 12
    "SLOPE"      # 13
]

assert len(CHANNEL_NAMES) == N_CHANNELS

assert len(set(CHANNEL_NAMES)) == N_CHANNELS

print("Project root:")
print(PROJECT_ROOT)

print("\nNumber of channels:", len(CHANNEL_NAMES))

print("\nFrozen channel order:")

for i, name in enumerate(CHANNEL_NAMES):

    print(
        f"{i:2d} : {name}"
    )

print("\nCell 1 completed.")

In [ ]:
# ============================================================
# Cell 2: Freeze physics-family membership
# ============================================================

PHYSICS_FAMILIES = {

    "magnetic": [
        "RTE_TMI"
    ],

    "gravity": [
        "CBG",
        "CBG_RES",
        "HGM"
    ],

    "radiometric": [
        "K",
        "eTh",
        "eU"
    ],

    "thermal": [
        "CPD"
    ],

    "structural": [
        "MAG_LD",
        "GRAV_LD",
        "DEM_LD",
        "ID"
    ],

    "terrain": [
        "DEM",
        "SLOPE"
    ]
}

# ------------------------------------------------------------
# Convert names to tensor-channel indices
# ------------------------------------------------------------

PHYSICS_FAMILY_INDICES = {}

for family, channels in PHYSICS_FAMILIES.items():

    PHYSICS_FAMILY_INDICES[family] = [

        CHANNEL_NAMES.index(channel)

        for channel in channels
    ]

# ------------------------------------------------------------
# Audit that every channel appears exactly once
# ------------------------------------------------------------

all_family_channels = [

    channel

    for channels in PHYSICS_FAMILIES.values()

    for channel in channels
]

assert len(all_family_channels) == N_CHANNELS

assert set(all_family_channels) == set(CHANNEL_NAMES)

assert len(set(all_family_channels)) == N_CHANNELS

# ------------------------------------------------------------
# Equal family weights
#
# Each physics family contributes 1/6 of the total
# reconstruction objective.
# ------------------------------------------------------------

N_FAMILIES = len(
    PHYSICS_FAMILIES
)

PHYSICS_FAMILY_WEIGHTS = {

    family:
        1.0 / N_FAMILIES

    for family in PHYSICS_FAMILIES
}

# ------------------------------------------------------------
# Build audit table
# ------------------------------------------------------------

family_records = []

for family, channels in PHYSICS_FAMILIES.items():

    indices = (
        PHYSICS_FAMILY_INDICES[
            family
        ]
    )

    family_records.append({

        "Physics_family":
            family,

        "N_channels":
            len(channels),

        "Channels":
            ", ".join(channels),

        "Tensor_indices":
            ", ".join(
                str(i)
                for i in indices
            ),

        "Family_loss_weight":
            PHYSICS_FAMILY_WEIGHTS[
                family
            ]
    })

physics_family_df = pd.DataFrame(
    family_records
)

display(
    physics_family_df
)

# ------------------------------------------------------------
# Save frozen configuration
# ------------------------------------------------------------

family_table_file = (
    TABLE_DIR /
    "Paper1_PBMMAE_physics_family_definition.csv"
)

physics_family_df.to_csv(
    family_table_file,
    index=False
)

print("\nPhysics families:", N_FAMILIES)

print(
    "Weight sum:",
    sum(
        PHYSICS_FAMILY_WEIGHTS.values()
    )
)

print("\nSaved:")
print(family_table_file)

print("\nCell 2 completed.")

In [ ]:
# ============================================================
# Cell 3: Audit real patch tensors by physics family
#          Representative Fold 3
#          9x9 and 13x13
# ============================================================

AUDIT_FOLD = 3

family_audit_records = []

for patch_size in PATCH_SIZES:

    print("\n" + "=" * 80)
    print(
        f"PATCH SIZE {patch_size}x{patch_size} — FOLD {AUDIT_FOLD}"
    )
    print("=" * 80)

    fold_dir = (
        PATCH_DIR /
        f"{patch_size}x{patch_size}" /
        f"Fold_{AUDIT_FOLD}"
    )

    train_file = (
        fold_dir /
        f"Fold{AUDIT_FOLD}_TRAIN_"
        f"{patch_size}x{patch_size}_14ch.npy"
    )

    val_file = (
        fold_dir /
        f"Fold{AUDIT_FOLD}_VALIDATION_"
        f"{patch_size}x{patch_size}_14ch.npy"
    )

    # --------------------------------------------------------
    # File existence
    # --------------------------------------------------------

    assert train_file.exists(), train_file
    assert val_file.exists(), val_file

    # --------------------------------------------------------
    # Load
    # Shape expected: (N, H, W, C)
    # --------------------------------------------------------

    X_train = np.load(
        train_file
    ).astype(np.float32)

    X_val = np.load(
        val_file
    ).astype(np.float32)

    print("TRAIN:", X_train.shape)
    print("VAL  :", X_val.shape)

    # --------------------------------------------------------
    # Geometry / channel assertions
    # --------------------------------------------------------

    assert X_train.ndim == 4
    assert X_val.ndim == 4

    assert X_train.shape[1] == patch_size
    assert X_train.shape[2] == patch_size
    assert X_train.shape[3] == N_CHANNELS

    assert X_val.shape[1] == patch_size
    assert X_val.shape[2] == patch_size
    assert X_val.shape[3] == N_CHANNELS

    assert np.isfinite(X_train).all()
    assert np.isfinite(X_val).all()

    # --------------------------------------------------------
    # Family-wise descriptive statistics
    # --------------------------------------------------------

    for family, indices in PHYSICS_FAMILY_INDICES.items():

        train_family = X_train[
            ...,
            indices
        ]

        val_family = X_val[
            ...,
            indices
        ]

        family_audit_records.append({

            "Patch_size":
                patch_size,

            "Fold":
                AUDIT_FOLD,

            "Physics_family":
                family,

            "N_channels":
                len(indices),

            "TRAIN_mean":
                float(
                    np.mean(train_family)
                ),

            "TRAIN_std":
                float(
                    np.std(
                        train_family,
                        ddof=1
                    )
                ),

            "TRAIN_median":
                float(
                    np.median(train_family)
                ),

            "TRAIN_abs_P95":
                float(
                    np.percentile(
                        np.abs(train_family),
                        95
                    )
                ),

            "TRAIN_abs_P99":
                float(
                    np.percentile(
                        np.abs(train_family),
                        99
                    )
                ),

            "TRAIN_abs_max":
                float(
                    np.max(
                        np.abs(train_family)
                    )
                ),

            "VAL_mean":
                float(
                    np.mean(val_family)
                ),

            "VAL_std":
                float(
                    np.std(
                        val_family,
                        ddof=1
                    )
                ),

            "VAL_median":
                float(
                    np.median(val_family)
                ),

            "VAL_abs_P95":
                float(
                    np.percentile(
                        np.abs(val_family),
                        95
                    )
                ),

            "VAL_abs_P99":
                float(
                    np.percentile(
                        np.abs(val_family),
                        99
                    )
                ),

            "VAL_abs_max":
                float(
                    np.max(
                        np.abs(val_family)
                    )
                )
        })

# ------------------------------------------------------------
# Build audit table
# ------------------------------------------------------------

family_audit_df = pd.DataFrame(
    family_audit_records
)

print("\nPHYSICS-FAMILY TENSOR AUDIT")
print("=" * 80)

display(
    family_audit_df.round(4)
)

# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

family_audit_file = (
    TABLE_DIR /
    "Paper1_PBMMAE_Fold3_physics_family_tensor_audit.csv"
)

family_audit_df.to_csv(
    family_audit_file,
    index=False
)

print("\nSaved:")
print(family_audit_file)

print("\nCell 3 completed.")

In [ ]:
# ============================================================
# Cell 4: Freeze PB-MMAE reconstruction-loss configuration
# ============================================================

LOSS_NAME = "SmoothL1"

SMOOTH_L1_BETA = 1.0

# Equal contribution from each of the six physics families
FAMILY_LOSS_WEIGHTS = {
    family: 1.0 / len(PHYSICS_FAMILIES)
    for family in PHYSICS_FAMILIES
}

loss_config_records = []

for family in PHYSICS_FAMILIES:

    loss_config_records.append({

        "Physics_family":
            family,

        "N_channels":
            len(
                PHYSICS_FAMILIES[
                    family
                ]
            ),

        "Loss":
            LOSS_NAME,

        "SmoothL1_beta":
            SMOOTH_L1_BETA,

        "Family_weight":
            FAMILY_LOSS_WEIGHTS[
                family
            ]
    })

loss_config_df = pd.DataFrame(
    loss_config_records
)

display(
    loss_config_df
)

assert np.isclose(
    sum(
        FAMILY_LOSS_WEIGHTS.values()
    ),
    1.0
)

loss_config_file = (
    TABLE_DIR /
    "Paper1_PBMMAE_reconstruction_loss_configuration.csv"
)

loss_config_df.to_csv(
    loss_config_file,
    index=False
)

print(
    "\nLoss:",
    LOSS_NAME
)

print(
    "Smooth L1 beta:",
    SMOOTH_L1_BETA
)

print(
    "Total family weight:",
    sum(
        FAMILY_LOSS_WEIGHTS.values()
    )
)

print("\nSaved:")
print(loss_config_file)

print("\nCell 4 completed.")

In [ ]:
# ============================================================
# Cell 5: Freeze PB-MMAE multimodal architecture specification
# ============================================================

# ------------------------------------------------------------
# Branch configuration
#
# Equal branch capacity across all six physics families.
# ------------------------------------------------------------

BRANCH_EMBED_DIM = 16

FUSION_INPUT_DIM = (
    len(PHYSICS_FAMILIES)
    * BRANCH_EMBED_DIM
)

FUSED_LATENT_DIM = 64

# CNN feature widths to be used in each family branch
BRANCH_CONV_CHANNELS = [16, 32]

# ------------------------------------------------------------
# Architecture inventory
# ------------------------------------------------------------

architecture_records = []

for family, channels in PHYSICS_FAMILIES.items():

    architecture_records.append({

        "Physics_family":
            family,

        "Input_channels":
            len(channels),

        "Channel_names":
            ", ".join(channels),

        "Conv_width_1":
            BRANCH_CONV_CHANNELS[0],

        "Conv_width_2":
            BRANCH_CONV_CHANNELS[1],

        "Branch_embedding_dim":
            BRANCH_EMBED_DIM,

        "Shared_fusion_input_dim":
            FUSION_INPUT_DIM,

        "Final_fused_latent_dim":
            FUSED_LATENT_DIM
    })

architecture_df = pd.DataFrame(
    architecture_records
)

display(
    architecture_df
)

# ------------------------------------------------------------
# Assertions
# ------------------------------------------------------------

assert (
    FUSION_INPUT_DIM ==
    6 * BRANCH_EMBED_DIM
)

assert (
    FUSION_INPUT_DIM == 96
)

assert (
    FUSED_LATENT_DIM == 64
)

assert (
    architecture_df[
        "Branch_embedding_dim"
    ].nunique() == 1
)

# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

architecture_file = (
    TABLE_DIR /
    "Paper1_PBMMAE_architecture_configuration.csv"
)

architecture_df.to_csv(
    architecture_file,
    index=False
)

print("\nPB-MMAE architecture specification")
print("=" * 60)

print(
    "Physics families:",
    len(PHYSICS_FAMILIES)
)

print(
    "Branch embedding dimension:",
    BRANCH_EMBED_DIM
)

print(
    "Concatenated multimodal dimension:",
    FUSION_INPUT_DIM
)

print(
    "Final fused latent dimension:",
    FUSED_LATENT_DIM
)

print(
    "Branch convolution widths:",
    BRANCH_CONV_CHANNELS
)

print("\nSaved:")
print(architecture_file)

print("\nCell 5 completed.")

In [ ]:
# ============================================================
# Cell 6: Freeze PB-MMAE masking protocol
# ============================================================

SPATIAL_MASK_RATIO = 0.40

MODALITY_MASK_PROB = 0.30

MAX_MASKED_FAMILIES = 1

N_PHYSICS_FAMILIES = len(
    PHYSICS_FAMILIES
)

# ------------------------------------------------------------
# Masking configuration table
# ------------------------------------------------------------

masking_config_df = pd.DataFrame(
    [
        {
            "Masking_component":
                "Spatial masking",

            "Primary_value":
                SPATIAL_MASK_RATIO,

            "Interpretation":
                "Randomly mask 40% of spatial positions "
                "within each available physics family"
        },

        {
            "Masking_component":
                "Whole-modality masking",

            "Primary_value":
                MODALITY_MASK_PROB,

            "Interpretation":
                "With 30% probability, completely mask "
                "one physics family for a training sample"
        },

        {
            "Masking_component":
                "Maximum masked families",

            "Primary_value":
                MAX_MASKED_FAMILIES,

            "Interpretation":
                "At most one of the six physics families "
                "is completely removed per training sample"
        }
    ]
)

display(
    masking_config_df
)

# ------------------------------------------------------------
# Assertions
# ------------------------------------------------------------

assert 0.0 < SPATIAL_MASK_RATIO < 1.0

assert 0.0 <= MODALITY_MASK_PROB <= 1.0

assert (
    MAX_MASKED_FAMILIES <
    N_PHYSICS_FAMILIES
)

# ------------------------------------------------------------
# Approximate visible spatial cells
# ------------------------------------------------------------

mask_geometry_records = []

for patch_size in PATCH_SIZES:

    total_cells = (
        patch_size *
        patch_size
    )

    masked_cells = int(
        round(
            total_cells *
            SPATIAL_MASK_RATIO
        )
    )

    visible_cells = (
        total_cells -
        masked_cells
    )

    mask_geometry_records.append(
        {
            "Patch_size":
                patch_size,

            "Total_spatial_cells":
                total_cells,

            "Approx_masked_cells":
                masked_cells,

            "Approx_visible_cells":
                visible_cells,

            "Masked_fraction":
                masked_cells /
                total_cells,

            "Visible_fraction":
                visible_cells /
                total_cells
        }
    )

mask_geometry_df = pd.DataFrame(
    mask_geometry_records
)

print(
    "\nSPATIAL MASK GEOMETRY"
)

display(
    mask_geometry_df.round(4)
)

# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

masking_config_file = (
    TABLE_DIR /
    "Paper1_PBMMAE_masking_configuration.csv"
)

mask_geometry_file = (
    TABLE_DIR /
    "Paper1_PBMMAE_mask_geometry.csv"
)

masking_config_df.to_csv(
    masking_config_file,
    index=False
)

mask_geometry_df.to_csv(
    mask_geometry_file,
    index=False
)

print("\nSaved:")
print(masking_config_file)
print(mask_geometry_file)

print("\nCell 6 completed.")

In [ ]:
# ============================================================
# Cell 7: NumPy prototype of PB-MMAE masking
#          Test on real Fold 3 patches
# ============================================================

# ------------------------------------------------------------
# Representative data
# ------------------------------------------------------------

TEST_PATCH_SIZE = 13
TEST_FOLD = 3

fold_dir = (
    PATCH_DIR /
    f"{TEST_PATCH_SIZE}x{TEST_PATCH_SIZE}" /
    f"Fold_{TEST_FOLD}"
)

train_file = (
    fold_dir /
    f"Fold{TEST_FOLD}_TRAIN_"
    f"{TEST_PATCH_SIZE}x{TEST_PATCH_SIZE}_14ch.npy"
)

X_train_masktest = np.load(
    train_file
).astype(np.float32)

print("TRAIN tensor:", X_train_masktest.shape)

# ------------------------------------------------------------
# Reproducible RNG
# ------------------------------------------------------------

MASK_TEST_SEED = 42

rng = np.random.default_rng(
    MASK_TEST_SEED
)

# ------------------------------------------------------------
# Mask generator
#
# Output:
#   masked_input
#   spatial_mask
#   family_mask
#
# spatial_mask:
#   True = spatial location deliberately hidden
#
# family_mask:
#   True = whole physics family deliberately hidden
# ------------------------------------------------------------

def apply_pbmmae_mask(
    patch,
    physics_family_indices,
    spatial_mask_ratio=0.40,
    modality_mask_prob=0.30,
    rng=None
):

    if rng is None:
        rng = np.random.default_rng()

    masked_patch = patch.copy()

    H, W, C = patch.shape

    # One spatial mask per physics family
    spatial_masks = {}

    # Whether the whole family is removed
    whole_family_masks = {
        family: False
        for family in physics_family_indices
    }

    # --------------------------------------------------------
    # Whole-modality masking
    # --------------------------------------------------------

    if rng.random() < modality_mask_prob:

        family_to_mask = rng.choice(
            list(
                physics_family_indices.keys()
            )
        )

        whole_family_masks[
            family_to_mask
        ] = True

    # --------------------------------------------------------
    # Spatial masking family-by-family
    # --------------------------------------------------------

    total_spatial_positions = H * W

    n_mask = int(
        round(
            total_spatial_positions *
            spatial_mask_ratio
        )
    )

    for family, indices in physics_family_indices.items():

        # ----------------------------------------------------
        # If entire family is masked:
        # set all its channels to zero.
        #
        # Zero is the mask token here because the input rasters
        # are robust-scaled around approximately zero.
        # ----------------------------------------------------

        if whole_family_masks[family]:

            masked_patch[
                ...,
                indices
            ] = 0.0

            spatial_masks[family] = np.ones(
                (H, W),
                dtype=bool
            )

            continue

        # ----------------------------------------------------
        # Otherwise generate random spatial mask
        # ----------------------------------------------------

        flat_indices = rng.choice(
            total_spatial_positions,
            size=n_mask,
            replace=False
        )

        family_mask = np.zeros(
            total_spatial_positions,
            dtype=bool
        )

        family_mask[
            flat_indices
        ] = True

        family_mask = family_mask.reshape(
            H,
            W
        )

        # Same spatial mask applied to every channel
        # belonging to this physics family.
        for channel_idx in indices:

            channel = masked_patch[
                ...,
                channel_idx
            ]

            channel[
                family_mask
            ] = 0.0

            masked_patch[
                ...,
                channel_idx
            ] = channel

        spatial_masks[
            family
        ] = family_mask

    return (
        masked_patch,
        spatial_masks,
        whole_family_masks
    )


# ------------------------------------------------------------
# Test on one real patch
# ------------------------------------------------------------

test_patch = X_train_masktest[0]

(
    masked_patch,
    spatial_masks,
    whole_family_masks
) = apply_pbmmae_mask(

    patch=test_patch,

    physics_family_indices=
        PHYSICS_FAMILY_INDICES,

    spatial_mask_ratio=
        SPATIAL_MASK_RATIO,

    modality_mask_prob=
        MODALITY_MASK_PROB,

    rng=rng
)

print(
    "\nOriginal patch shape:",
    test_patch.shape
)

print(
    "Masked patch shape:",
    masked_patch.shape
)

print(
    "Original finite:",
    np.isfinite(
        test_patch
    ).all()
)

print(
    "Masked finite:",
    np.isfinite(
        masked_patch
    ).all()
)

print("\nWhole-family masking:")

for family, status in whole_family_masks.items():

    print(
        f"{family:12s}: {status}"
    )


# ------------------------------------------------------------
# Mask-count audit
# ------------------------------------------------------------

mask_count_records = []

for family, family_mask in spatial_masks.items():

    n_masked = int(
        family_mask.sum()
    )

    total = int(
        family_mask.size
    )

    mask_count_records.append({

        "Physics_family":
            family,

        "Whole_family_masked":
            whole_family_masks[
                family
            ],

        "Masked_positions":
            n_masked,

        "Total_positions":
            total,

        "Masked_fraction":
            n_masked / total
    })

mask_test_df = pd.DataFrame(
    mask_count_records
)

print("\nMASK AUDIT")

display(
    mask_test_df
)


# ------------------------------------------------------------
# Assertions
# ------------------------------------------------------------

assert (
    masked_patch.shape ==
    test_patch.shape
)

assert np.isfinite(
    masked_patch
).all()

assert (
    sum(
        whole_family_masks.values()
    )
    <= 1
)

for family in PHYSICS_FAMILY_INDICES:

    if whole_family_masks[family]:

        assert np.all(
            masked_patch[
                ...,
                PHYSICS_FAMILY_INDICES[
                    family
                ]
            ] == 0.0
        )

print(
    "\nAll masking assertions PASSED."
)

# ------------------------------------------------------------
# Save audit
# ------------------------------------------------------------

mask_test_file = (
    TABLE_DIR /
    "Paper1_PBMMAE_masking_unit_test.csv"
)

mask_test_df.to_csv(
    mask_test_file,
    index=False
)

print("\nSaved:")
print(mask_test_file)

print("\nCell 7 completed.")

In [ ]:
# ============================================================
# Cell 8: Empirical masking-frequency audit
#          Many Fold 3, 13x13 patches
# ============================================================

N_MASK_AUDIT_SAMPLES = 1000
MASK_AUDIT_SEED = 12345

rng = np.random.default_rng(
    MASK_AUDIT_SEED
)

whole_mask_count = 0

family_whole_mask_counts = {
    family: 0
    for family in PHYSICS_FAMILY_INDICES
}

spatial_fraction_records = []

n_available_patches = len(
    X_train_masktest
)

for i in range(
    N_MASK_AUDIT_SAMPLES
):

    # --------------------------------------------------------
    # Cycle through available patches
    # --------------------------------------------------------

    patch = X_train_masktest[
        i % n_available_patches
    ]

    (
        masked_patch,
        spatial_masks,
        whole_family_masks
    ) = apply_pbmmae_mask(

        patch=patch,

        physics_family_indices=
            PHYSICS_FAMILY_INDICES,

        spatial_mask_ratio=
            SPATIAL_MASK_RATIO,

        modality_mask_prob=
            MODALITY_MASK_PROB,

        rng=rng
    )

    # --------------------------------------------------------
    # Whole-modality audit
    # --------------------------------------------------------

    n_whole = int(
        sum(
            whole_family_masks.values()
        )
    )

    whole_mask_count += n_whole

    for family, status in whole_family_masks.items():

        if status:

            family_whole_mask_counts[
                family
            ] += 1

    # --------------------------------------------------------
    # Spatial mask audit
    #
    # Only count families that were not fully removed.
    # --------------------------------------------------------

    for family, mask in spatial_masks.items():

        if not whole_family_masks[
            family
        ]:

            spatial_fraction_records.append({

                "Sample":
                    i,

                "Physics_family":
                    family,

                "Spatial_mask_fraction":
                    float(
                        mask.mean()
                    )
            })


# ------------------------------------------------------------
# Whole-modality results
# ------------------------------------------------------------

empirical_modality_prob = (
    whole_mask_count /
    N_MASK_AUDIT_SAMPLES
)

print(
    "Requested whole-modality probability:",
    MODALITY_MASK_PROB
)

print(
    "Empirical whole-modality probability:",
    round(
        empirical_modality_prob,
        4
    )
)

print(
    "Whole-modality masked samples:",
    whole_mask_count,
    "/",
    N_MASK_AUDIT_SAMPLES
)


# ------------------------------------------------------------
# Family-specific whole-mask frequencies
# ------------------------------------------------------------

family_mask_frequency_df = pd.DataFrame(
    [
        {
            "Physics_family":
                family,

            "Whole_mask_count":
                count,

            "Frequency_all_samples":
                count /
                N_MASK_AUDIT_SAMPLES,

            "Expected_if_uniform":
                MODALITY_MASK_PROB /
                len(
                    PHYSICS_FAMILY_INDICES
                )
        }

        for family, count
        in family_whole_mask_counts.items()
    ]
)

print(
    "\nWHOLE-MODALITY FAMILY FREQUENCY"
)

display(
    family_mask_frequency_df.round(4)
)


# ------------------------------------------------------------
# Spatial mask results
# ------------------------------------------------------------

spatial_fraction_df = pd.DataFrame(
    spatial_fraction_records
)

spatial_summary_df = (
    spatial_fraction_df
    .groupby(
        "Physics_family"
    )
    .agg(
        N_observations=(
            "Spatial_mask_fraction",
            "size"
        ),

        Mean_spatial_mask_fraction=(
            "Spatial_mask_fraction",
            "mean"
        ),

        SD_spatial_mask_fraction=(
            "Spatial_mask_fraction",
            "std"
        ),

        Min_spatial_mask_fraction=(
            "Spatial_mask_fraction",
            "min"
        ),

        Max_spatial_mask_fraction=(
            "Spatial_mask_fraction",
            "max"
        )
    )
    .reset_index()
)

print(
    "\nSPATIAL MASKING SUMMARY"
)

display(
    spatial_summary_df.round(4)
)


# ------------------------------------------------------------
# Assertions
# ------------------------------------------------------------

# Whole-modality probability should be reasonably close
# to the target over 1000 random draws.

assert abs(
    empirical_modality_prob
    - MODALITY_MASK_PROB
) < 0.05

# Every non-whole-masked family should use the same
# deterministic spatial fraction for a given patch size.

expected_spatial_fraction = (
    round(
        TEST_PATCH_SIZE
        * TEST_PATCH_SIZE
        * SPATIAL_MASK_RATIO
    )
    /
    (
        TEST_PATCH_SIZE
        * TEST_PATCH_SIZE
    )
)

assert np.allclose(
    spatial_fraction_df[
        "Spatial_mask_fraction"
    ].values,
    expected_spatial_fraction
)

print(
    "\nExpected realized spatial fraction:",
    round(
        expected_spatial_fraction,
        4
    )
)

print(
    "All empirical masking checks PASSED."
)


# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

whole_mask_file = (
    TABLE_DIR /
    "Paper1_PBMMAE_empirical_modality_mask_frequency.csv"
)

spatial_mask_file = (
    TABLE_DIR /
    "Paper1_PBMMAE_empirical_spatial_mask_summary.csv"
)

family_mask_frequency_df.to_csv(
    whole_mask_file,
    index=False
)

spatial_summary_df.to_csv(
    spatial_mask_file,
    index=False
)

print("\nSaved:")
print(whole_mask_file)
print(spatial_mask_file)

print("\nCell 8 completed.")

In [ ]:
# ============================================================
# Cell 9: PB-MMAE fused latent KMeans sweep
#         Fold 3, 13x13
#         k = 2 ... 10
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.cluster import KMeans
from sklearn.metrics import (
    silhouette_score,
    davies_bouldin_score,
    calinski_harabasz_score
)

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

PROJECT_ROOT = Path(
    r"C:\project 3\Paper1_SSL"
)

PBMMAE_RUN_DIR = (
    PROJECT_ROOT /
    "03_outputs" /
    "models" /
    "PBMMAE" /
    "Fold3_13x13"
)

train_latent_file = (
    PBMMAE_RUN_DIR /
    "PBMMAE_Fold3_13x13_TRAIN_latent64.npy"
)

val_latent_file = (
    PBMMAE_RUN_DIR /
    "PBMMAE_Fold3_13x13_VALIDATION_latent64.npy"
)

# ------------------------------------------------------------
# Load fused embeddings
# ------------------------------------------------------------

Z_train_pbm = np.load(
    train_latent_file
)

Z_val_pbm = np.load(
    val_latent_file
)

print("TRAIN latent:", Z_train_pbm.shape)
print("VALIDATION latent:", Z_val_pbm.shape)

assert Z_train_pbm.shape[1] == 64
assert Z_val_pbm.shape[1] == 64

assert np.isfinite(
    Z_train_pbm
).all()

assert np.isfinite(
    Z_val_pbm
).all()

# ------------------------------------------------------------
# KMeans sweep
# ------------------------------------------------------------

K_VALUES = range(2, 11)

records = []

for k in K_VALUES:

    km = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=50,
        algorithm="lloyd"
    )

    train_labels = km.fit_predict(
        Z_train_pbm
    )

    val_labels = km.predict(
        Z_val_pbm
    )

    # --------------------------------------------------------
    # Internal TRAIN metrics
    # --------------------------------------------------------

    sil = silhouette_score(
        Z_train_pbm,
        train_labels
    )

    dbi = davies_bouldin_score(
        Z_train_pbm,
        train_labels
    )

    ch = calinski_harabasz_score(
        Z_train_pbm,
        train_labels
    )

    # --------------------------------------------------------
    # Occupancy
    # --------------------------------------------------------

    train_counts = np.bincount(
        train_labels,
        minlength=k
    )

    val_counts = np.bincount(
        val_labels,
        minlength=k
    )

    empty_val = int(
        np.sum(
            val_counts == 0
        )
    )

    records.append({
        "k": k,

        "Silhouette":
            sil,

        "Davies_Bouldin":
            dbi,

        "Calinski_Harabasz":
            ch,

        "TRAIN_min_cluster":
            int(
                train_counts.min()
            ),

        "TRAIN_max_cluster":
            int(
                train_counts.max()
            ),

        "VALIDATION_min_cluster":
            int(
                val_counts.min()
            ),

        "VALIDATION_max_cluster":
            int(
                val_counts.max()
            ),

        "Empty_validation_clusters":
            empty_val
    })

    print(
        f"k={k:2d} | "
        f"Sil={sil:.4f} | "
        f"DB={dbi:.4f} | "
        f"CH={ch:.2f} | "
        f"TRAIN min={train_counts.min():3d} | "
        f"VAL min={val_counts.min():3d} | "
        f"Empty VAL={empty_val}"
    )

# ------------------------------------------------------------
# Table
# ------------------------------------------------------------

pbm_k_sweep_df = pd.DataFrame(
    records
)

print("\nPB-MMAE KMEANS SWEEP")
print("=" * 80)

display(
    pbm_k_sweep_df.round(4)
)

# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

output_file = (
    PBMMAE_RUN_DIR /
    "PBMMAE_Fold3_13x13_KMeans_k2to10.csv"
)

pbm_k_sweep_df.to_csv(
    output_file,
    index=False
)

print("\nSaved to:")
print(output_file)

print("\nCell 9 completed.")

In [ ]:
# ============================================================
# Cell 10: PB-MMAE consensus-clustering diagnostic
#          Fold 3, 13x13
#
# Repeated KMeans -> co-association matrix ->
# hierarchical consensus partition
#
# IMPORTANT:
# Consensus is built using TRAIN only.
# VALIDATION is assigned afterward using TRAIN consensus
# centroids in the fixed 64-D PB-MMAE latent space.
# ============================================================

from itertools import combinations

import numpy as np
import pandas as pd

from sklearn.cluster import (
    KMeans,
    AgglomerativeClustering
)

from sklearn.metrics import (
    adjusted_rand_score,
    normalized_mutual_info_score,
    silhouette_score,
    davies_bouldin_score,
    calinski_harabasz_score
)

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

CONSENSUS_K_VALUES = range(2, 11)

N_CONSENSUS_RUNS = 50

CONSENSUS_BASE_SEED = 1000

print("TRAIN latent:", Z_train_pbm.shape)
print("VALIDATION latent:", Z_val_pbm.shape)

assert Z_train_pbm.shape == (455, 64)
assert Z_val_pbm.shape == (392, 64)

# ------------------------------------------------------------
# Helper: pairwise initialization stability
# ------------------------------------------------------------

def pairwise_partition_stability(
    label_runs
):

    ari_values = []
    nmi_values = []

    for i, j in combinations(
        range(len(label_runs)),
        2
    ):

        ari_values.append(
            adjusted_rand_score(
                label_runs[i],
                label_runs[j]
            )
        )

        nmi_values.append(
            normalized_mutual_info_score(
                label_runs[i],
                label_runs[j]
            )
        )

    return {
        "ARI_mean":
            float(np.mean(ari_values)),

        "ARI_std":
            float(
                np.std(
                    ari_values,
                    ddof=1
                )
            ),

        "ARI_min":
            float(np.min(ari_values)),

        "ARI_median":
            float(np.median(ari_values)),

        "NMI_mean":
            float(np.mean(nmi_values)),

        "NMI_std":
            float(
                np.std(
                    nmi_values,
                    ddof=1
                )
            ),

        "NMI_min":
            float(np.min(nmi_values)),

        "NMI_median":
            float(np.median(nmi_values))
    }


# ------------------------------------------------------------
# Helper: co-association matrix
#
# Entry (i,j) = fraction of repeated KMeans runs
# in which TRAIN samples i and j belong to same cluster.
# ------------------------------------------------------------

def build_coassociation_matrix(
    label_runs
):

    n_samples = len(
        label_runs[0]
    )

    coassoc = np.zeros(
        (
            n_samples,
            n_samples
        ),
        dtype=np.float32
    )

    for labels in label_runs:

        same_cluster = (
            labels[:, None]
            ==
            labels[None, :]
        )

        coassoc += (
            same_cluster.astype(
                np.float32
            )
        )

    coassoc /= len(
        label_runs
    )

    np.fill_diagonal(
        coassoc,
        1.0
    )

    return coassoc


# ------------------------------------------------------------
# Helper: PAC
#
# Proportion of ambiguous pairwise consensus values.
# Lower PAC = more stable consensus.
#
# We use the conventional broad ambiguity interval
# 0.10 < consensus < 0.90.
# ------------------------------------------------------------

def calculate_pac(
    coassoc,
    lower=0.10,
    upper=0.90
):

    upper_triangle = (
        coassoc[
            np.triu_indices_from(
                coassoc,
                k=1
            )
        ]
    )

    ambiguous = (
        (upper_triangle > lower)
        &
        (upper_triangle < upper)
    )

    return float(
        np.mean(
            ambiguous
        )
    )


# ------------------------------------------------------------
# Helper: consensus clustering
# ------------------------------------------------------------

def consensus_partition(
    coassoc,
    k
):

    distance_matrix = (
        1.0 -
        coassoc
    )

    np.fill_diagonal(
        distance_matrix,
        0.0
    )

    # Compatible with newer and older sklearn versions.
    try:

        model = AgglomerativeClustering(
            n_clusters=k,
            metric="precomputed",
            linkage="average"
        )

    except TypeError:

        model = AgglomerativeClustering(
            n_clusters=k,
            affinity="precomputed",
            linkage="average"
        )

    return model.fit_predict(
        distance_matrix
    )


# ------------------------------------------------------------
# Helper: TRAIN consensus centroids
# ------------------------------------------------------------

def calculate_consensus_centroids(
    Z_train,
    labels,
    k
):

    centroids = []

    for cluster_id in range(k):

        cluster_points = Z_train[
            labels == cluster_id
        ]

        assert len(
            cluster_points
        ) > 0

        centroids.append(
            cluster_points.mean(
                axis=0
            )
        )

    return np.vstack(
        centroids
    )


# ------------------------------------------------------------
# Helper: assign validation to TRAIN consensus domains
# ------------------------------------------------------------

def assign_to_centroids(
    Z,
    centroids
):

    # squared Euclidean distances
    distances = (
        (
            Z[:, None, :]
            -
            centroids[None, :, :]
        )
        ** 2
    ).sum(
        axis=2
    )

    return np.argmin(
        distances,
        axis=1
    )


# ============================================================
# Run k = 2 ... 10
# ============================================================

consensus_records = []

consensus_objects = {}

for k in CONSENSUS_K_VALUES:

    print(
        "\n" + "=" * 78
    )

    print(
        f"CONSENSUS k={k}"
    )

    print(
        "=" * 78
    )

    # --------------------------------------------------------
    # 1. Repeated KMeans on TRAIN only
    # --------------------------------------------------------

    label_runs = []

    for run in range(
        N_CONSENSUS_RUNS
    ):

        km = KMeans(
            n_clusters=k,

            random_state=(
                CONSENSUS_BASE_SEED
                + run
            ),

            # single initialization intentionally:
            # variation across runs is what we are measuring
            n_init=1,

            algorithm="lloyd"
        )

        labels = km.fit_predict(
            Z_train_pbm
        )

        label_runs.append(
            labels
        )

    # --------------------------------------------------------
    # 2. Initialization stability
    # --------------------------------------------------------

    stability = (
        pairwise_partition_stability(
            label_runs
        )
    )

    # --------------------------------------------------------
    # 3. Co-association matrix
    # --------------------------------------------------------

    coassoc = (
        build_coassociation_matrix(
            label_runs
        )
    )

    pac = calculate_pac(
        coassoc
    )

    # --------------------------------------------------------
    # 4. Consensus TRAIN partition
    # --------------------------------------------------------

    consensus_train_labels = (
        consensus_partition(
            coassoc,
            k
        )
    )

    # --------------------------------------------------------
    # 5. Internal metrics for consensus TRAIN labels
    # --------------------------------------------------------

    sil = silhouette_score(
        Z_train_pbm,
        consensus_train_labels
    )

    dbi = davies_bouldin_score(
        Z_train_pbm,
        consensus_train_labels
    )

    ch = calinski_harabasz_score(
        Z_train_pbm,
        consensus_train_labels
    )

    # --------------------------------------------------------
    # 6. TRAIN consensus occupancy
    # --------------------------------------------------------

    train_counts = np.bincount(
        consensus_train_labels,
        minlength=k
    )

    # --------------------------------------------------------
    # 7. Consensus centroids derived from TRAIN only
    # --------------------------------------------------------

    consensus_centroids = (
        calculate_consensus_centroids(
            Z_train_pbm,
            consensus_train_labels,
            k
        )
    )

    # --------------------------------------------------------
    # 8. Assign held-out VALIDATION
    # --------------------------------------------------------

    consensus_val_labels = (
        assign_to_centroids(
            Z_val_pbm,
            consensus_centroids
        )
    )

    val_counts = np.bincount(
        consensus_val_labels,
        minlength=k
    )

    empty_val = int(
        np.sum(
            val_counts == 0
        )
    )

    # --------------------------------------------------------
    # 9. Within-consensus strength
    #
    # Mean co-association of sample pairs that ended up
    # in the same consensus domain.
    # --------------------------------------------------------

    within_values = []

    for cluster_id in range(k):

        idx = np.where(
            consensus_train_labels
            == cluster_id
        )[0]

        if len(idx) > 1:

            submatrix = coassoc[
                np.ix_(
                    idx,
                    idx
                )
            ]

            vals = submatrix[
                np.triu_indices_from(
                    submatrix,
                    k=1
                )
            ]

            within_values.extend(
                vals.tolist()
            )

    mean_within_consensus = (
        float(
            np.mean(
                within_values
            )
        )
        if len(within_values) > 0
        else np.nan
    )

    # --------------------------------------------------------
    # Record
    # --------------------------------------------------------

    record = {

        "k":
            k,

        "Runs":
            N_CONSENSUS_RUNS,

        "ARI_mean":
            stability[
                "ARI_mean"
            ],

        "ARI_std":
            stability[
                "ARI_std"
            ],

        "ARI_min":
            stability[
                "ARI_min"
            ],

        "ARI_median":
            stability[
                "ARI_median"
            ],

        "NMI_mean":
            stability[
                "NMI_mean"
            ],

        "NMI_std":
            stability[
                "NMI_std"
            ],

        "NMI_min":
            stability[
                "NMI_min"
            ],

        "NMI_median":
            stability[
                "NMI_median"
            ],

        "PAC_0.1_0.9":
            pac,

        "Mean_within_consensus":
            mean_within_consensus,

        "Silhouette":
            sil,

        "Davies_Bouldin":
            dbi,

        "Calinski_Harabasz":
            ch,

        "TRAIN_min_cluster":
            int(
                train_counts.min()
            ),

        "TRAIN_max_cluster":
            int(
                train_counts.max()
            ),

        "VALIDATION_min_cluster":
            int(
                val_counts.min()
            ),

        "VALIDATION_max_cluster":
            int(
                val_counts.max()
            ),

        "Empty_validation_clusters":
            empty_val
    }

    consensus_records.append(
        record
    )

    consensus_objects[k] = {

        "coassociation":
            coassoc,

        "train_labels":
            consensus_train_labels,

        "validation_labels":
            consensus_val_labels,

        "centroids":
            consensus_centroids
    }

    # --------------------------------------------------------
    # Screen summary
    # --------------------------------------------------------

    print(
        f"ARI mean/median = "
        f"{stability['ARI_mean']:.3f} / "
        f"{stability['ARI_median']:.3f}"
    )

    print(
        f"NMI mean/median = "
        f"{stability['NMI_mean']:.3f} / "
        f"{stability['NMI_median']:.3f}"
    )

    print(
        f"PAC = {pac:.3f}"
    )

    print(
        f"Within consensus = "
        f"{mean_within_consensus:.3f}"
    )

    print(
        f"Sil={sil:.3f} | "
        f"DBI={dbi:.3f} | "
        f"CH={ch:.1f}"
    )

    print(
        "TRAIN counts:",
        train_counts.tolist()
    )

    print(
        "VALIDATION counts:",
        val_counts.tolist()
    )

    print(
        "Empty validation domains:",
        empty_val
    )


# ============================================================
# Results table
# ============================================================

pbm_consensus_df = pd.DataFrame(
    consensus_records
)

print(
    "\nPB-MMAE CONSENSUS CLUSTERING RESULTS"
)

print(
    "=" * 90
)

display(
    pbm_consensus_df.round(
        4
    )
)


# ============================================================
# Save summary
# ============================================================

consensus_summary_file = (
    PBMMAE_RUN_DIR /
    "PBMMAE_Fold3_13x13_consensus_k2to10.csv"
)

pbm_consensus_df.to_csv(
    consensus_summary_file,
    index=False
)


# ============================================================
# Save labels + co-association matrices
# ============================================================

for k in CONSENSUS_K_VALUES:

    obj = consensus_objects[
        k
    ]

    np.save(
        PBMMAE_RUN_DIR /
        f"PBMMAE_Fold3_13x13_"
        f"consensus_k{k}_TRAIN_labels.npy",

        obj[
            "train_labels"
        ]
    )

    np.save(
        PBMMAE_RUN_DIR /
        f"PBMMAE_Fold3_13x13_"
        f"consensus_k{k}_VALIDATION_labels.npy",

        obj[
            "validation_labels"
        ]
    )

    np.save(
        PBMMAE_RUN_DIR /
        f"PBMMAE_Fold3_13x13_"
        f"consensus_k{k}_coassociation.npy",

        obj[
            "coassociation"
        ]
    )


print(
    "\nSaved:"
)

print(
    consensus_summary_file
)

print(
    "\nCell 10 completed."
)